In [1]:
import pandas as pd
from pathlib import Path
import holidays
import numpy as np

In [34]:
ROOT = Path.cwd().resolve().parent

DATA = ROOT / "data/toptur"

expediciones = DATA / "Mayo26/expediciones.xls"
frecuencias_fijas = DATA / "POT_V_VALPARAISOUN07_UN07_NORMAL_2026_A1_2.xlsx"

## Expediciones

In [35]:
df_exp = pd.read_html(expediciones, flavor = 'lxml')[0]

In [36]:
df_exp["Fecha"] = pd.to_datetime(df_exp["Fecha"])
anios = df_exp["Fecha"].dt.year.unique().tolist()
feriados_cl = holidays.Chile(years = anios)

In [37]:
es_feriado = df_exp['Fecha'].apply(lambda x: x in feriados_cl)
es_domingo = df_exp['Fecha'].dt.weekday == 6
es_sabado = df_exp['Fecha'].dt.weekday == 5

condiciones = [
    es_feriado | es_domingo,
    es_sabado
]

valores = ['DF', 'DS']

df_exp["tipo_dia"] = np.select(condiciones, valores, default = 'DL')

In [ ]:
df_exp["sentido_str"] = df_exp["Sentido"].astype(str).str.strip().str.upper().str[0]
df_exp["servicio_str"] = df_exp["Servicio"].astype(str).str.strip()
df_exp["periodo_num"] = pd.to_numeric(df_exp["Periodo"], errors='coerce').astype('Int64')

In [39]:
filtro_valida = df_exp["Estado"].astype(str).str.strip() == "Valida"
df_exp_validas = df_exp[filtro_valida].copy()

In [40]:
df_exp_validas.head(2)

,Fecha,Inicio Expedicion,Fin Expedicion,Folio TS,ID Exp,Bus,Chofer,Propietario,Variante,Servicio,...,10,11,12,13,14,15,tipo_dia,sentido_str,servicio_str,periodo_num
2,2026-05-01,2026-05-01 06:59:52,2026-05-01 07:42:29,2605017310420,6886345,116/FRDH12,PRUEBA PRUEBA,GLADYS MONTOLLA .,731,701,...,07:31:51,07:36:12,07:37:48,07:38:46,07:42:29,NaN,DF,I,701,6
3,2026-05-01,2026-05-01 07:00:34,2026-05-01 07:46:32,2605017150420,6886371,153/GWXF15,JUAN LUIS ZUÃIGA BRAVO,NaN,715,705,...,07:32:44,07:35:00,07:39:39,07:41:42,07:45:22,07:46:32,DF,I,705,7


In [41]:
df_conteo = df_exp_validas.groupby(
    ['Fecha', 'servicio_str', 'sentido_str', 'tipo_dia', 'periodo_num']
).size().reset_index(name="expediciones_observadas")

df_conteo.rename(columns={
    'servicio_str': 'servicio', 
    'sentido_str': 'sentido', 
    'periodo_num': 'periodo' # Lo volvemos a llamar periodo para el cruce
}, inplace=True)

df_conteo.head(2)

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas
0,2026-05-01,701,I,DF,6,1
1,2026-05-01,701,I,DF,7,2


In [42]:
df_conteo.tail(2)

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas
5467,2026-05-31,706,R,DF,18,1
5468,2026-05-31,706,R,DF,19,2


In [43]:
df_conteo[df_conteo["periodo"] == 11]

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas
5,2026-05-01,701,I,DF,11,2
19,2026-05-01,701,R,DF,11,1
33,2026-05-01,702,I,DF,11,1
45,2026-05-01,702,R,DF,11,2
55,2026-05-01,703,I,DF,11,1
...,...,...,...,...,...,...
5406,2026-05-31,704,R,DF,11,2
5420,2026-05-31,705,I,DF,11,4
5433,2026-05-31,705,R,DF,11,3
5447,2026-05-31,706,I,DF,11,2


In [47]:
df_conteo[(df_conteo["Fecha"] == "2026-05-10") & (df_conteo["sentido"] == "I") & (df_conteo["periodo"] == "19")]

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas


## Frecuencias A1

In [16]:
excel_completo = pd.ExcelFile(frecuencias_fijas)
hojas_servicios = [hoja for hoja in excel_completo.sheet_names if '-' in hoja]

lista_dataframes = []

for hoja in hojas_servicios:
    # Extraemos el código del servicio y el sentido desde el nombre de la pestaña
    # Ej: '701-I' -> servicio='701', sentido='I'
    servicio, sentido = hoja.split('-')
    
    
    df_frec = pd.read_excel(excel_completo, sheet_name=hoja, skiprows=10, header=[0, 1])
    df_frec = df_frec.dropna(axis=1, how='all')
    df_frec = df_frec[df_frec.iloc[:, 0].astype(str).str.strip().str.lower() != 'total']
    
    if df_frec.empty:
        continue
        
    # Asignar las columnas aplanadas
    df_frec.columns = [
        'periodo', 'horario', 
        'demanda_DL', 'frecuencia_DL', 
        'demanda_DS', 'frecuencia_DS', 
        'demanda_DF', 'frecuencia_DF'
    ]
    
    # Separar y etiquetar por tipo de día
    # Laboral (DL)
    df_dl = df_frec[['periodo', 'demanda_DL', 'frecuencia_DL']].copy()
    df_dl['tipo_dia'] = 'DL'
    df_dl.rename(columns={'demanda_DL': 'tipo_demanda', 'frecuencia_DL': 'frecuencia'}, inplace=True)

    # Sábado (DS)
    df_ds = df_frec[['periodo', 'demanda_DS', 'frecuencia_DS']].copy()
    df_ds['tipo_dia'] = 'DS'
    df_ds.rename(columns={'demanda_DS': 'tipo_demanda', 'frecuencia_DS': 'frecuencia'}, inplace=True)

    # Domingo / Festivo (DF)
    df_df = df_frec[['periodo', 'demanda_DF', 'frecuencia_DF']].copy()
    df_df['tipo_dia'] = 'DF'
    df_df.rename(columns={'demanda_DF': 'tipo_demanda', 'frecuencia_DF': 'frecuencia'}, inplace=True)
    
    # Apilar los días de la pestaña actual
    df_hoja_normalizado = pd.concat([df_dl, df_ds, df_df], ignore_index=True)
    
    # Agregar las columnas de identificación del servicio
    df_hoja_normalizado['servicio'] = servicio
    df_hoja_normalizado['sentido'] = sentido
    
    lista_dataframes.append(df_hoja_normalizado)

In [17]:
df_master_frecuencias = pd.concat(lista_dataframes, ignore_index=True)
df_master_frecuencias = df_master_frecuencias[[
    'servicio', 'sentido', 'periodo', 'tipo_dia', 'tipo_demanda', 'frecuencia'
]]

df_master_frecuencias["periodo"] = pd.to_numeric(df_master_frecuencias["periodo"], errors='coerce').astype('Int64')

In [18]:
# df_a1 = df_master_frecuencias[df_master_frecuencias["tipo_demanda"].notna()]

df_a1 = df_master_frecuencias[df_master_frecuencias["tipo_demanda"].notna()].copy()
df_a1 = df_a1.rename(columns={"frecuencia": "EE"})

calendario = df_exp[['Fecha', 'tipo_dia']].drop_duplicates()
df_base_exigida = pd.merge(calendario, df_a1, on='tipo_dia', how='left')

In [20]:
df_a1.head(3)

,servicio,sentido,periodo,tipo_dia,tipo_demanda,frecuencia
6,701,I,6,DL,ALTA,8.0
7,701,I,7,DL,ALTA,8.0
8,701,I,8,DL,ALTA,7.0


In [19]:
df_a1.tail(2)

,servicio,sentido,periodo,tipo_dia,tipo_demanda,frecuencia
859,706,R,19,DF,MEDIA,3.0
860,706,R,20,DF,MEDIA,3.0


In [ ]:
# df_a1 = df_a1.rename(columns={"frecuencia": "frecuencia_a1"})

## Calculo ICF

In [37]:
chequeo = pd.merge(
    df_conteo, df_a1,
    on=['servicio', 'sentido', 'periodo', 'tipo_dia'],
    how='left', indicator=True
)
fuera_de_norma = chequeo[chequeo['_merge'] == 'left_only']
print(f"Expediciones observadas sin exigencia asociada: {len(fuera_de_norma)}")
fuera_de_norma[['servicio','sentido','periodo','tipo_dia']].drop_duplicates()

Expediciones observadas sin exigencia asociada: 93


,servicio,sentido,periodo,tipo_dia
0,701,I,6,DF
65,703,R,7,DF
144,706,R,7,DF
231,703,R,6,DS
262,704,R,6,DS
293,705,I,21,DS
324,706,R,6,DS
421,703,R,21,DF
436,704,R,7,DF
612,704,R,6,DL


In [19]:
# df_icf = pd.merge(
#     df_a1,
#     df_conteo,
#     on=["servicio","sentido","periodo","tipo_dia"],
#     how = "left"
# )

# df_icf = df_icf.rename(columns = {
#     "expediciones_observadas": "EO",
#     "frecuencia_a1": "EE"
# })

df_icf = pd.merge(
    df_base_exigida,
    df_conteo,
    on=['Fecha', 'servicio', 'sentido', 'tipo_dia', 'periodo'],
    how='left'
)

df_icf = df_icf.rename(columns={
    "expediciones_observadas": "EO"
})

df_icf["EO"] = df_icf["EO"].fillna(0)

df_icf['ICF'] = np.floor(np.minimum(df_icf['EE'], df_icf['EO']) / df_icf['EE'] * 100 + 0.5) / 100

In [20]:
df_icf.head(2)

,Fecha,tipo_dia,servicio,sentido,periodo,tipo_demanda,EE,EO,ICF
0,2026-04-01,DL,701,I,6,ALTA,8.0,8.0,1.00
1,2026-04-01,DL,701,I,7,ALTA,8.0,2.0,0.25


In [15]:
df_icf[(df_icf["Fecha"] == "2026-05-01") & (df_icf["servicio"] == "701")& (df_icf["periodo"] == 7)]

,Fecha,tipo_dia,servicio,sentido,periodo,tipo_demanda,EE,EO,ICF
0,2026-05-01,DF,701,I,7,MEDIA,5.0,2.0,0.4
14,2026-05-01,DF,701,R,7,BAJA,2.0,1.0,0.5


In [16]:
df_icf.ICF.unique().tolist()

[0.4,
 0.2,
 0.6,
 0.25,
 0.5,
 0.75,
 0.0,
 0.67,
 0.33,
 1.0,
 0.17,
 0.86,
 0.71,
 0.83,
 0.8,
 0.43,
 0.57,
 0.29,
 0.88,
 0.56,
 0.78,
 0.63,
 0.9,
 0.89,
 0.44,
 0.7,
 0.14]

In [21]:
def calcular_psi(df,fecha_columna="Fecha",fecha_inicio_operacion=None,mas_de_24_meses=None):
    """
    Calcula el parámetro psi (ψ) según la antigüedad de la operación,
    de acuerdo a lo indicado en la Res. 49/2024 (pág. 41):
        - Hasta el mes 24 de operación: psi = 0,90
        - Desde el mes 25 en adelante:  psi = 0,95

    Parámetros
    ----------
    df : DataFrame
        DataFrame que contiene la columna de fecha sobre la que se calculará psi.
    fecha_columna : str
        Nombre de la columna de fecha en df (default 'Fecha').
    fecha_inicio_operacion : str o Timestamp, opcional
        Fecha de inicio de operación del Perímetro de Exclusión. Si se entrega,
        psi se calcula automáticamente fila por fila según la antigüedad real
        a la fecha de cada registro.
    mas_de_24_meses : bool, opcional
        Si no quieres calcular la antigüedad real (por ejemplo, para otros
        servicios donde solo sabes "ya superó los 24 meses" o no), puedes
        forzar el valor directamente: True -> psi=0.95 para todo el df,
        False -> psi=0.90 para todo el df.

    Debes entregar UNO de los dos parámetros: fecha_inicio_operacion o mas_de_24_meses.

    Retorna
    -------
    Series con el valor de psi para cada fila de df.
    """
    if fecha_inicio_operacion is None and mas_de_24_meses is None:
        raise ValueError(
            "Debes especificar 'fecha_inicio_operacion' o 'mas_de_24_meses'."
        )

    if mas_de_24_meses is not None:
        # Modo simple: se fuerza el mismo psi para todo el dataframe
        return pd.Series(0.95 if mas_de_24_meses else 0.90, index=df.index)

    # Modo automático: calcular antigüedad real fila por fila
    fecha_inicio_operacion = pd.Timestamp(fecha_inicio_operacion)

    def mes_operacion(fecha):
        return (fecha.year - fecha_inicio_operacion.year) * 12 + (fecha.month - fecha_inicio_operacion.month) + 1

    meses_operacion = df[fecha_columna].apply(mes_operacion)
    return np.where(meses_operacion <= 24, 0.90, 0.95)

In [22]:
def tabla_periodo_vs_fecha(df_icf, servicio, sentido):
    """
    Reproduce la tabla 'periodo x fecha' (con promedio final) usando la
    razón cruda EO/EE (SIN aplicar el min/cap del ICF normativo) — esto
    es lo que muestra el reporte de referencia, no el ICF oficial.
    """
    df_filtro = df_icf[
        (df_icf['servicio'] == servicio) &
        (df_icf['sentido'] == sentido)
    ].copy()

    # Razón cruda, sin el cap de min(EE,EO)
    df_filtro['ratio_crudo'] = df_filtro['EO'] / df_filtro['EE']

    tabla = df_filtro.pivot_table(
        index='periodo',
        columns='Fecha',
        values='ratio_crudo',
        aggfunc='mean'   # por si hay más de una fila por periodo-fecha (no debería)
    )

    # Formatear columnas de fecha igual al reporte (YYYY-MM-DD)
    tabla.columns = [c.strftime('%Y-%m-%d') for c in tabla.columns]

    # Agregar columna de promedio simple por periodo (a través de todas las fechas)
    tabla['Promedio'] = tabla.mean(axis=1).round(2)

    return tabla

In [23]:
df_icf["psi"] = calcular_psi(df_icf, mas_de_24_meses=True)
#df_icf_otro['psi'] = calcular_psi(df_icf_otro, fecha_inicio_operacion='2025-03-15')
#df_icf_otro['psi'] = calcular_psi(df_icf_otro, mas_de_24_meses=False)

In [24]:
df_icf

,Fecha,tipo_dia,servicio,sentido,periodo,tipo_demanda,EE,EO,ICF,psi
0,2026-04-01,DL,701,I,6,ALTA,8.0,8.0,1.00,0.95
1,2026-04-01,DL,701,I,7,ALTA,8.0,2.0,0.25,0.95
2,2026-04-01,DL,701,I,8,ALTA,7.0,2.0,0.29,0.95
3,2026-04-01,DL,701,I,9,ALTA,7.0,5.0,0.71,0.95
4,2026-04-01,DL,701,I,10,ALTA,6.0,5.0,0.83,0.95
...,...,...,...,...,...,...,...,...,...,...
5395,2026-04-30,DL,706,R,17,ALTA,5.0,2.0,0.40,0.95
5396,2026-04-30,DL,706,R,18,ALTA,5.0,3.0,0.60,0.95
5397,2026-04-30,DL,706,R,19,ALTA,4.0,1.0,0.25,0.95
5398,2026-04-30,DL,706,R,20,MEDIA,3.0,0.0,0.00,0.95


In [25]:
df_grouped = df_icf.groupby(
    ['servicio', 'sentido', 'tipo_demanda'],
    as_index=False
).agg(
    ICF_promedio=('ICF', 'mean'),
    psi=('psi', 'first')   # asumiendo que psi no cambia dentro del mismo mes
)

df_grouped['ICF_promedio'] = df_grouped['ICF_promedio'].round(2)
df_grouped

,servicio,sentido,tipo_demanda,ICF_promedio,psi
0,701,I,ALTA,0.93,0.95
1,701,I,BAJA,1.00,0.95
2,701,I,MEDIA,0.87,0.95
3,701,R,ALTA,0.92,0.95
4,701,R,BAJA,0.73,0.95
5,701,R,MEDIA,0.88,0.95
6,702,I,ALTA,0.77,0.95
7,702,I,BAJA,0.44,0.95
8,702,I,MEDIA,0.81,0.95
9,702,R,ALTA,0.68,0.95


In [26]:
df_grouped.to_excel("./revision1.xlsx")

In [22]:
CARPETA_REPORTES = ROOT / "salidas" / "reporte_servicio_sentido"
CARPETA_REPORTES.mkdir(parents = True, exist_ok=True)

In [23]:
combinaciones = df_icf[['servicio', 'sentido']].drop_duplicates().sort_values(['servicio','sentido'])
resumen_tipo_demanda = []

In [25]:
for _, row in combinaciones.iterrows():
    servicio = row['servicio']
    sentido = row['sentido']

    # --- 1. Tabla detallada periodo x fecha (la de diagnóstico, ratio crudo) ---
    tabla = tabla_periodo_vs_fecha(df_icf, servicio=servicio, sentido=sentido)

    nombre_archivo = f"{servicio}_{sentido}.xlsx"
    tabla.to_excel(CARPETA_REPORTES / nombre_archivo)

    # --- 2. Promedio por tipo_demanda (el ICF normativo, capeado con min) ---
    df_filtro = df_icf[(df_icf['servicio'] == servicio) & (df_icf['sentido'] == sentido)]

    promedios = df_filtro.groupby('tipo_demanda', as_index=False).agg(
        ICF_promedio=('ICF', 'mean')
    )
    promedios['ICF_promedio'] = promedios['ICF_promedio'].round(2)
    promedios['servicio'] = servicio
    promedios['sentido'] = sentido

    resumen_tipo_demanda.append(promedios)

In [26]:
df_resumen_tipo_demanda = pd.concat(resumen_tipo_demanda, ignore_index=True)
df_resumen_tipo_demanda = df_resumen_tipo_demanda[['servicio', 'sentido', 'tipo_demanda', 'ICF_promedio']]

In [27]:
df_resumen_tipo_demanda

,servicio,sentido,tipo_demanda,ICF_promedio
0,701,I,ALTA,0.89
1,701,I,BAJA,0.93
2,701,I,MEDIA,0.78
3,701,R,ALTA,0.89
4,701,R,BAJA,0.73
5,701,R,MEDIA,0.80
6,702,I,ALTA,0.71
7,702,I,BAJA,0.46
8,702,I,MEDIA,0.71
9,702,R,ALTA,0.63


In [31]:
def aplicar_regla_pago(icf, psi=0.95):
    if icf < 0.50:
        return 0.50
    elif icf > psi:
        return 1.00
    else:
        return icf

In [32]:
df_resumen_tipo_demanda["factor_pago"] = df_resumen_tipo_demanda["ICF_promedio"].apply(
    lambda x: aplicar_regla_pago(x, psi=0.95) # Usamos 0.95 porque pusiste mas_de_24_meses=True
)

In [33]:
df_ranking_final = df_resumen_tipo_demanda.groupby("servicio")["factor_pago"].mean().reset_index()

df_ranking_final.rename(columns = {
    "factor_pago":"ICF"
}, inplace = True)

df_ranking_final["ICF"] = df_ranking_final["ICF"].round(2)

In [34]:
df_ranking_final

,servicio,ICF
0,701,0.84
1,702,0.62
2,703,0.55
3,704,0.70
4,705,0.69
5,706,0.62
